# Animating the flow at a fixed time

Notebook 02 drew storm-relative streamlines through a RAMS updraft, with
cones that point downstream. This notebook animates them. The storm stays
frozen at one output time. Only the cones move along the streamlines, at
the speed of the air.

It covers:

1. how the cones move: travel time and the flow clock;
2. choosing the spacing of the cones from the wind speeds;
3. a *hold*: one model time rendered over many video frames;
4. rendering one loop and repeating it in the video.

Run it in the pixi `blender` environment, after notebook 02.

In [ ]:
import numpy as np

import notebook_setup as setup
import skyvista as sv

from cloud_render_config import make_cloud_export_config
from rams_lite_loader import load_rams_lite_hydrometeors, load_rams_lite_winds
from render_rams_streamlines_side_by_side import pinned_view, storm_motion_estimate

## 1. How the cones move

Each point on a streamline has a *travel time*: the time air moving with
the wind takes to get there from the streamline's seed. It is the integral
of distance divided by speed along the line. It is negative upstream of the
seed and positive downstream.

With `arrow_travel_time = T`, the cones sit where the travel time is
`clock + k * T`, for every whole number k. The *flow clock* is a time in
seconds that advances from video frame to video frame. When it advances by
some amount, every cone moves downstream as far as the air moves in that
time. Cones therefore move fast and far apart where the wind is strong, and
slowly and close together where it is weak.

Where slow air brings the cones closer than two cone lengths apart, they
shrink in proportion. Without that, overlapping cones would merge into what
looks like a thick tube. Cone size therefore also shows the wind speed.

After the clock advances by exactly T, every cone has reached the place of
its downstream neighbour. The picture is then identical to the first
frame, so one loop of T is all that needs rendering.

In [ ]:
# The storm at 94 min: hydrometeors and cell-centered wind.
storm_clouds = load_rams_lite_hydrometeors(setup.RAMS_STORM_FILE, top_height=16500.0)
storm_winds = load_rams_lite_winds(setup.RAMS_STORM_FILE, top_height=16500.0)
storm_minutes = float(storm_clouds["time"]) / 60.0

# Storm-relative streamlines seeded in the updraft, as in notebook 02.
storm_motion = storm_motion_estimate(storm_winds)
storm_winds["in_updraft"] = storm_winds["W"] > 5.0
STREAMLINE_W_LIMIT = 12.0


def make_updraft_streamlines(arrow_travel_time):
    """Storm-relative streamlines through the updraft, with moving cones."""
    return sv.make_streamlines(
        u="U", v="V", w="W",
        seed_stride=24,
        seed_mask_by="in_updraft",
        max_length=25000.0,
        min_length=2000.0,
        point_spacing=100.0,
        frame_velocity=storm_motion,
        color_by="w",
        cmap="RdBu_r",
        clim=(-STREAMLINE_W_LIMIT, STREAMLINE_W_LIMIT),
        tube_radius=45.0,
        arrow_travel_time=arrow_travel_time,
        shader="self_lit",
    )

## 2. Choosing the spacing of the cones

The spacing between neighbouring cones is the local wind speed times T. A
good T puts the cones about 2 km apart at a typical speed.

`polylines` traces the streamlines in Python and returns each line's points,
travel times and speeds. The speeds here are storm-relative, so they are
lower than the ground-relative wind.

In [ ]:
traced_lines = make_updraft_streamlines(arrow_travel_time=None).polylines(storm_winds, None)
all_speeds = np.concatenate([line.speeds for line in traced_lines])
speed_percentiles = np.percentile(all_speeds, [10, 50, 90])
print(f"{len(traced_lines)} streamlines")
print("storm-relative speed, 10th / 50th / 90th percentile:",
      " / ".join(f"{speed:.1f}" for speed in speed_percentiles), "m/s")

# About 2 km between cones at the median speed.
CONE_SPACING_AT_MEDIAN_SPEED = 2000.0
arrow_travel_time = float(np.round(CONE_SPACING_AT_MEDIAN_SPEED / speed_percentiles[1], -1))
print(f"cones every {arrow_travel_time:.0f} s of travel")

## 3. The hold

`BlenderHoldConfig` renders a scene that has one data time over many video
frames:

- `n_frames`: the number of video frames.
- `clock_seconds_per_frame`: how far the flow clock advances per frame (s).

One loop advances the clock by exactly T, so
`clock_seconds_per_frame = T / n_frames`.

The motion must stay smooth. If the clock advanced by more than about a
quarter of T per frame, the eye could not match each cone with its
position in the next frame. The cones would then seem to stand still or
run backwards, like wagon wheels on film. With 48 frames per loop, each
frame advances T / 48.

The camera is the automatic framing of the plain cloud, as in notebook 02.
Here the cloud is only exported, not rendered, because only its camera is
needed.

In [ ]:
from types import SimpleNamespace

# Video frames per loop, and the video frame rate.
FRAMES_PER_LOOP = 48
VIDEO_FPS = 24.0


def make_preview_config(origin_shift=None):
    """The settings of notebook 02's panels, at preview quality."""
    arguments = SimpleNamespace(
        width=800, height=600, samples=setup.PREVIEW_SAMPLES,
        sun_elevation=30.0, sun_azimuth=15.0,
        camera_azimuth=-55.0, camera_elevation=-8.0, camera_distance=1.1,
    )
    return make_cloud_export_config(arguments, origin_shift=origin_shift)


# The automatic camera of the plain cloud (export only, no render).
framing_bundle_dir = setup.WORK_DIR / "flow_framing"
sv.Scene().add_clouds(storm_clouds, microphysics="rams").to_blender(
    framing_bundle_dir, config=make_preview_config()
)
camera_location, camera_look_at, origin_shift = pinned_view(framing_bundle_dir)

flow_config = make_preview_config(origin_shift=origin_shift)
flow_config.camera.location = camera_location
flow_config.camera.look_at = camera_look_at
flow_config.overlay_colorbars = False
flow_config.fps = VIDEO_FPS
# One loop: the clock advances one marker travel time over the hold.
flow_config.hold = sv.BlenderHoldConfig(
    n_frames=FRAMES_PER_LOOP,
    clock_seconds_per_frame=arrow_travel_time / FRAMES_PER_LOOP,
)
speed_up = arrow_travel_time / FRAMES_PER_LOOP * VIDEO_FPS
print(f"the flow plays {speed_up:.0f}x faster than real time")

In [ ]:
flow_bundle_dir = setup.WORK_DIR / "flow_hold"
flow_scene = sv.Scene().add_clouds(storm_clouds, microphysics="rams", extinction_scale=0.03)
flow_scene.add(storm_winds, make_updraft_streamlines(arrow_travel_time))
flow_scene.to_blender(flow_bundle_dir, config=flow_config, render=True, quiet=True)
print(f"rendered {FRAMES_PER_LOOP} frames to {flow_bundle_dir / 'render'}")

Two frames a quarter loop apart show the motion. The crop below is the
upper-level outflow, on the right of the storm. Every cone has moved a
quarter of the way to its downstream neighbour.

In [ ]:
from PIL import Image

# The outflow region of the 800 x 600 frames: (left, top, right, bottom) px.
OUTFLOW_CROP_BOX = (440, 100, 720, 310)
# Enlargement of the crop, so the cones are easy to see.
CROP_ENLARGEMENT = 2


def cropped_frame(frame_number):
    """The outflow region of one rendered frame, enlarged, as an array."""
    with Image.open(setup.rendered_frame(flow_bundle_dir, frame_number)) as frame_image:
        crop = frame_image.convert("RGB").crop(OUTFLOW_CROP_BOX)
    enlarged_size = (crop.width * CROP_ENLARGEMENT, crop.height * CROP_ENLARGEMENT)
    return np.asarray(crop.resize(enlarged_size, Image.LANCZOS))


quarter_loop_frame = 1 + FRAMES_PER_LOOP // 4
motion_pair_path = setup.WORK_DIR / "flow_motion_pair.png"
sv.compose.Figure(
    panels=[
        sv.compose.Panel(
            cropped_frame(frame_number),
            [sv.compose.Text(f"Video frame {frame_number}", size=0.05)],
        )
        for frame_number in (1, quarter_loop_frame)
    ]
).render(motion_pair_path)
setup.show_image(motion_pair_path, width=1000)

## 4. Labels, and the loop repeated in the video

The labels go on every frame with `Figure.render(path, FrameContext(...))`.
`sv.encode_video` then writes the MP4. Its `repeat` argument plays the
48-frame loop several times without rendering it again.

In [ ]:
# Text heights, as fractions of the frame height.
TITLE_SIZE, LABEL_SIZE = 0.04, 0.03

flow_figure = sv.compose.Figure(
    panels=[
        sv.compose.Panel(
            lambda frame: frame.render_path,
            [
                sv.compose.Text("Storm-relative streamlines through the updraft", size=TITLE_SIZE),
                sv.compose.Text(
                    f"t = {storm_minutes:.1f} min (held)", size=LABEL_SIZE, offset=(0.0, -0.055)
                ),
                sv.compose.Text(
                    f"Cones move with the air, {speed_up:.0f}x real time\n"
                    f"Cone spacing: {arrow_travel_time:.0f} s of travel",
                    anchor="lower left",
                    size=LABEL_SIZE,
                ),
                sv.compose.Colorbar(
                    cmap="RdBu_r",
                    clim=(-STREAMLINE_W_LIMIT, STREAMLINE_W_LIMIT),
                    label="Vertical velocity w (m/s)",
                    ticks=list(np.linspace(-STREAMLINE_W_LIMIT, STREAMLINE_W_LIMIT, 5)),
                    anchor="lower right",
                    length=0.3,
                    text_size=LABEL_SIZE,
                ),
            ],
        )
    ]
)

# Label every rendered frame.
composed_dir = setup.WORK_DIR / "flow_composed"
composed_dir.mkdir(exist_ok=True)
for frame_index in range(FRAMES_PER_LOOP):
    flow_figure.render(
        composed_dir / f"frame_{frame_index + 1:04d}.png",
        sv.compose.FrameContext(
            index=frame_index,
            n_frames=FRAMES_PER_LOOP,
            render_path=setup.rendered_frame(flow_bundle_dir, frame_index + 1),
        ),
    )

# Five loops: 10 s of video.
flow_video = sv.encode_video(
    composed_dir / "frame_%04d.png",
    setup.WORK_DIR / "flow_hold.mp4",
    fps=VIDEO_FPS,
    repeat=5,
    crf=18,
)
setup.show_video(flow_video)

## Where to go from here

- `examples/rams_clouds/animate_rams_streamlines.py` makes this animation
  from the command line, at 1440 x 1080. Its output is
  `examples/rams_clouds/outputs/streamline_animation.mp4`.
- Inside a `BlenderAnimation` (notebook 04), the flow clock is each frame's
  model time. Moving cones then advance with the time-lapse, so they show
  the flow at the speed the animation runs.